# 05 - Calendar events & service exceptions

Catalogs date-level calendar signals derived from SEPTA's GTFS releases. Fetches Philly home game schedules from 4 major league sports APIs.

Overview: SEPTA's `calendar_dates.txt` records planned service exceptions -- dates where a normal service
pattern is canceled (`exception_type = 2`) and/or a special pattern is added (`exception_type = 1`).
Unplanned disruptions are not captured here, and SEPTA does not release historical real-time service
updates.

Each (`date`, `service_id`) pair has exactly one `exception_type`. The pairing of a removal (type = 2) and
an addition (type = 1) on the same date always involves two different `service_id`s: the pattern being
canceled and its replacement, respectively. We see these "swaps" most commonly throughout the GTFS data (as
opposed to a service expansion or reduction alone).

SEPTA doesn't use distinct exception IDs per line until 2025; most across all years are holiday swaps across whole network (4b); 2025 they start releasing line-specific excepts.
Applied to each applicable section using gtfs crosswalk. Fall back to system wide only if one can't be resolved

Inputs:
- `2_gtfs_calendar_raw.parquet` (built by `02_gtfs_full_fetch.ipynb`): raw (`date`, `service_id`,
  `exception_type`, `gtfs_release_date`) rows across all historical GTFS releases.
- `2_gtfs_linkages_since_2017_clean.parquet` (built by `02_gtfs_full_fetch.ipynb`, also used by
  `02b_gtfs_merge.ipynb`): `service_id` -> `line` crosswalk per GTFS release, used in Section 4b to resolve
  which line(s) each exception actually affects.
- Home game schedules: Phillies, Eagles, Sixers, Flyers home game dates fetched from MLB, NHL, ESPN APIs

Outputs:
- `5_gtfs_calendar_dedup.parquet` - is `2_gtfs_calendar_raw.parquet` deduplicated and restricted to analysis window
- `5_home_games.parquet` - all Eagles, Sixers, Phillies, and Flyers home games (date, time, team) 2017-2025
- `5_calendar_by_date.parquet` - one row per calendar date with systemwide GTFS service exceptions
  (`service_additions`, `service_removals`) and PA public holiday flag (`is_holiday`); merged onto the
  reduced dataset in `07_reduce_for_modeling.ipynb` rather than the full row-level OTP dataset.
- `5_calendar_by_date_line.parquet` - one row per (date, line) with line-specific GTFS service
  exceptions (`service_additions_line`, `service_removals_line`); merged onto `07`'s modeling table by
  `(service_date, line)` and summed with the system-wide counts above for the final feature.

## 0. Imports and setup

In [1]:
import time
import holidays
import pandas as pd
import requests
from datetime import date, datetime, timedelta

BASEPATH = "../data"

YEAR_MIN = 2017
YEAR_MAX = 2025

## 1. Load `calendar_dates.txt` across all historical GTFS releases

In [2]:
# raw calendar_dates.txt rows across all releases, built by 02_gtfs_full_fetch.ipynb
df_raw = pd.read_parquet(f"{BASEPATH}/2_gtfs_calendar_raw.parquet")

## 2. Clean, deduplicate raw data
The same (`date`, `service_id`) pair can appear across multiple GTFS releases since SEPTA publishes rolling
updates. Here, keep the earliest appearance of each pair.

Checked this against `2_gtfs_calendar_raw.parquet` directly -- 269/371 pairs appear across multi releases so dedup req'd, b ut `exception_type` is only inconsistent across releases for 1 pair. keeping the latest release since I think most likely to be the corrected version 

In [ ]:
print(f"rows before dedup: {len(df_raw):,}")

# same date/service_id combo can appear across multiple 
# releases — keep="last"
df_raw["date"] = pd.to_datetime(df_raw["date"])
df = (
    df_raw
    .sort_values("gtfs_release_date")
    .drop_duplicates(subset = ["date", "service_id"],
                     keep = "last")
    .sort_values("date")
    .reset_index(drop = True)
    )
print(f"rows after dedup:  {len(df):,}")

# restrict to analysis window
df = df[df["date"].dt.year.between(YEAR_MIN, YEAR_MAX)].reset_index(drop = True)
print(f"rows in {YEAR_MIN}-{YEAR_MAX}: {len(df):,}")

# save dedup'd
df.to_parquet(f"{BASEPATH}/5_gtfs_calendar_dedup.parquet", index = False)

# view
df.head()

rows before dedup: 1,315
rows after dedup:  371
rows in 2017-2025: 245


,date,service_id,exception_type,gtfs_release_date
0,2017-01-02,M3,1,2016-12-16
1,2017-01-02,M1,2,2016-12-16
2,2017-05-29,M4,1,2017-08-23
3,2017-05-29,M1,2,2017-08-23
4,2017-07-04,M4,1,2017-08-23


In [4]:
df = pd.read_parquet(f"{BASEPATH}/5_gtfs_calendar_dedup.parquet")

## 3. Aggregate to date level

Counts service additions and removals per date. Kept as separate cols instead of binary flag because 3
distinct patterns exist:
- **Swap** (additions > 0, removals > 0): holiday service replacing normal service (most common)
- **Expansion** (additions > 0, removals == 0): extra service added on top of normal schedule
- **Reduction** (removals > 0, additions == 0): service cancelled with no replacement

In [5]:
df_agg = (
    df
    .groupby(["date", "exception_type"])
    .size()
    .unstack(fill_value = 0)
    .rename(columns = {1: "service_additions", 2: "service_removals"})
    .reset_index()
    )

# ensure both columns exist even if one exception_type never appears
for col in ["service_additions", "service_removals"]:
    if col not in df_agg.columns:
        df_agg[col] = 0

print(f"{len(df_agg):,} date rows")
df_agg.head(10)

91 date rows


exception_type,date,service_additions,service_removals
0,2017-01-02,1,1
1,2017-05-29,1,1
2,2017-07-04,1,1
3,2017-11-23,1,1
4,2018-05-28,1,1
5,2018-07-04,2,1
6,2018-09-03,1,1
7,2018-11-22,1,1
8,2018-12-25,1,1
9,2019-01-01,1,1


In [6]:
print(len(df_agg))
print(df_agg["date"].dt.year.value_counts().sort_index())

91
date
2017     4
2018     5
2019     6
2020     7
2021    11
2022    10
2023     6
2024     7
2025    35
Name: count, dtype: int64


Only 91 dates with service exceptions, and the number recorded spikes in 2025. This may be due to a shift
in SEPTA behavior/choices for GTFS releases + schedule exception filing. 91 dates across 8 years is pretty
sparse -- supplement with/compare to PA and federal holidays below.

## 4. Holidays
Using `holidays` package

In [7]:
years = range(YEAR_MIN, YEAR_MAX + 1)
pa_holidays = holidays.country_holidays("US",
                                        subdiv = "PA",
                                        years = years)
holiday_df = pd.DataFrame(
    [(pd.to_datetime(d), name) for d, name in pa_holidays.items()],
    columns = ["date", "holiday_name"]
).sort_values("date").reset_index(drop = True)

print(f"{len(holiday_df)} holiday dates found")
holiday_df.head(10)

116 holiday dates found


,date,holiday_name
0,2017-01-01,New Year's Day
1,2017-01-02,New Year's Day (observed)
2,2017-01-16,Martin Luther King Jr. Day
3,2017-02-20,Presidents' Day
4,2017-05-29,Memorial Day
5,2017-07-04,Independence Day
6,2017-09-04,Labor Day
7,2017-10-09,Columbus Day
8,2017-11-10,Veterans Day (observed)
9,2017-11-11,Veterans Day


In [8]:
holiday_df.tail(10)

,date,holiday_name
106,2025-02-17,Presidents' Day
107,2025-05-26,Memorial Day
108,2025-06-19,Juneteenth National Independence Day
109,2025-07-04,Independence Day
110,2025-09-01,Labor Day
111,2025-10-13,Columbus Day
112,2025-11-11,Veterans Day
113,2025-11-27,Thanksgiving Day
114,2025-11-28,Day After Thanksgiving
115,2025-12-25,Christmas Day


In [9]:
merged_check = holiday_df.merge(
    df_agg[["date", "service_additions", "service_removals"]],
    on  = "date",
    how = "outer",
    indicator = True
)

# 62 dates are holiday-only
# 54 dates are in both holidays and GTFS
# 37 dates are in GTFS only; we see them below
print(merged_check["_merge"].value_counts())
print()
# print(merged_check.sort_values("date"))

_merge
left_only     62
both          54
right_only    37
Name: count, dtype: int64



- 62 dates are holiday-only (no GTFS exceptions)
- 54 dates are in both holidays and GTFS
- 37 dates are in GTFS only; briefly examined below

In [10]:
print(merged_check[merged_check["_merge"] == "right_only"].sort_values("date").to_string())

          date holiday_name  service_additions  service_removals      _merge
44  2020-10-24          NaN                1.0               1.0  right_only
52  2021-05-23          NaN                1.0               1.0  right_only
58  2021-09-04          NaN                1.0               1.0  right_only
59  2021-09-05          NaN                1.0               1.0  right_only
70  2022-02-18          NaN                1.0               1.0  right_only
71  2022-02-19          NaN                1.0               1.0  right_only
72  2022-02-20          NaN                1.0               1.0  right_only
85  2022-12-31          NaN                1.0               0.0  right_only
112 2024-12-31          NaN                1.0               0.0  right_only
114 2025-01-11          NaN                1.0               1.0  right_only
115 2025-01-12          NaN                1.0               1.0  right_only
116 2025-01-18          NaN                1.0               1.0  right_only

Most of these dates are in 2025; it definitely seems like SEPTA may have changed their GTFS reporting last
year given how much of an uptick we see in 2025.
Also some grouping in these dates -- isolated weekends/weekdays, which might reflect planned track work or
maintenance.

Still merge the GTFS exception data onto the OTP data here, but might ultimately want to drop it from final
models. The GTFS set is sparse and majority-holiday, so unlikely to add much useful/independent signal
beyond `is_holiday`.

## 4b. Resolve exceptions to specific lines where possible

Checks each exception's `service_id` against `2_gtfs_linkages_since_2017_clean.parquet` (the crosswalk
built for `02b_gtfs_merge.ipynb`), joined on the exact GTFS release that reported the exception via
`gtfs_release_date`/`gtfs_date`.
Most exceptions resolve to systemwide; consistent with holidays. But some (conc. in 2025) resolve to just one or two specific lines - Fox Chase/Paoli and Lansdale/Media; maybe consistent w track work or closures.
Septa does seem to have changed some reporting patterns in 2025 gtfs; this is one (line-specific)

Exceptions are split into **broad** (kept systemwide) and **specific** (only applied to the line(s) actually affected). One weird release doesnt resolve (2022, odd service_id scheme) and is kept rbroad

In [11]:
import numpy as np

crosswalk = pd.read_parquet(f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet")
crosswalk["line"] = crosswalk["line"].str.replace(" Line$", "", regex = True)
# pseudo-lines aren't real Regional Rail service -- exclude from line-resolution
crosswalk = crosswalk[~crosswalk["line"].isin(["Center City Only Trains", "Airport Line Bus Substitution"])]
REAL_LINE_COUNT = crosswalk["line"].nunique()
print(f"{REAL_LINE_COUNT} real Regional Rail lines in crosswalk")

# per (service_id, exact release) line sets
sid_release_lines = crosswalk.groupby(["service_id", "gtfs_date"])["line"].agg(
    lambda s: sorted(set(s))
)
# fallback for the exception releases with no exact-release crosswalk match:
# pool across every release that service_id appears in at all
sid_pooled_lines = crosswalk.groupby("service_id")["line"].agg(lambda s: sorted(set(s)))


def resolve_lines(row):
    key = (row["service_id"], row["gtfs_release_date"])
    if key in sid_release_lines.index:
        return sid_release_lines.loc[key]
    return sid_pooled_lines.get(row["service_id"], [])


df["_lines"] = df.apply(resolve_lines, axis = 1)
df["_n_lines"] = df["_lines"].apply(len)

BROAD_THRESHOLD = 12  # of REAL_LINE_COUNT (14)
df["_scope"] = np.where(
    (df["_n_lines"] == 0) | (df["_n_lines"] >= BROAD_THRESHOLD), "broad", "specific",
)
print(df["_scope"].value_counts())
print(f"rows unresolved against crosswalk entirely (kept broad): {(df['_n_lines'] == 0).sum():,}")

14 real Regional Rail lines in crosswalk


_scope
broad       169
specific     76
Name: count, dtype: int64
rows unresolved against crosswalk entirely (kept broad): 14


In [12]:
# broad (system-wide) exceptions: same date-level aggregation as df_agg above,
# but restricted to the broad-scope rows -- this is what actually feeds the
# final calendar_by_date table below
df_broad = df[df["_scope"] == "broad"]
df_agg_broad = (
    df_broad
    .groupby(["date", "exception_type"])
    .size()
    .unstack(fill_value = 0)
    .rename(columns = {1: "service_additions", 2: "service_removals"})
    .reset_index()
)
for col in ["service_additions", "service_removals"]:
    if col not in df_agg_broad.columns:
        df_agg_broad[col] = 0
print(f"{len(df_agg_broad):,} broad (system-wide) date rows "
      f"(vs. {len(df_agg):,} before the line-specific split)")

# line-specific exceptions: explode to one row per (date, line), then
# aggregate to (date, line, exception_type)
df_specific = df[df["_scope"] == "specific"].explode("_lines").rename(columns = {"_lines": "line"})
calendar_by_date_line = (
    df_specific
    .groupby(["date", "line", "exception_type"])
    .size()
    .unstack(fill_value = 0)
    .rename(columns = {1: "service_additions_line", 2: "service_removals_line"})
    .reset_index()
    .rename(columns = {"date": "service_date"})
)
for col in ["service_additions_line", "service_removals_line"]:
    if col not in calendar_by_date_line.columns:
        calendar_by_date_line[col] = 0

calendar_by_date_line.to_parquet(f"{BASEPATH}/5_calendar_by_date_line.parquet", index = False)
print(f"{len(calendar_by_date_line):,} (date, line) rows saved to 5_calendar_by_date_line.parquet")
print(f"distinct dates affected: {calendar_by_date_line['service_date'].nunique():,}, "
      f"distinct lines affected: {calendar_by_date_line['line'].nunique():,}")
calendar_by_date_line.sort_values("service_date").head(10)

76 broad (system-wide) date rows (vs. 91 before the line-specific split)
309 (date, line) rows saved to 5_calendar_by_date_line.parquet
distinct dates affected: 31, distinct lines affected: 13


exception_type,service_date,line,service_additions_line,service_removals_line
0,2018-07-04,Chestnut Hill East,1,0
10,2018-07-04,Wilmington/Newark,1,0
9,2018-07-04,West Trenton,1,0
7,2018-07-04,Trenton,1,0
6,2018-07-04,Paoli/Thorndale,1,0
8,2018-07-04,Warminster,1,0
4,2018-07-04,Manayunk/Norristown,1,0
3,2018-07-04,Lansdale/Doylestown,1,0
2,2018-07-04,Fox Chase,1,0
1,2018-07-04,Chestnut Hill West,1,0


## 5. Build per-date calendar/holiday lookup table

Combines `df_agg` (GTFS service exceptions) and `holiday_df` (PA public holidays) into one small per-date
table.

In [13]:
calendar_by_date = df_agg_broad.merge(
    holiday_df[["date", "holiday_name"]],
    on = "date",
    how = "outer"
)

calendar_by_date["service_additions"] = calendar_by_date["service_additions"].fillna(0).astype(int)
calendar_by_date["service_removals"] = calendar_by_date["service_removals"].fillna(0).astype(int)
calendar_by_date["is_holiday"] = calendar_by_date["holiday_name"].notna().astype(int)
calendar_by_date = (calendar_by_date
    .drop(columns = "holiday_name")
    .rename(columns = {"date": "service_date"})
)

calendar_by_date.to_parquet(f"{BASEPATH}/5_calendar_by_date.parquet", index = False)
print(f"{len(calendar_by_date):,} date rows saved")
calendar_by_date.head()

142 date rows saved


,service_date,service_additions,service_removals,is_holiday
0,2017-01-01,0,0,1
1,2017-01-02,1,1,1
2,2017-01-16,0,0,1
3,2017-02-20,0,0,1
4,2017-05-29,1,1,1


## 6. Philadelphia sports schedules
Sources:
- Phillies / MLB: MLB Stats API
- Flyers / NHL: NHL API
- Eagles / NFL: ESPN unofficial API
- Sixers / NBA: ESPN unofficial API

Output: `df_sports` -- one row per game

In [14]:
# Setup
from zoneinfo import ZoneInfo
eastern = ZoneInfo("America/New_York")
SEASONS = range(YEAR_MIN, YEAR_MAX)

# General helper
def getter(url, **kwargs):
    """ Returns parsed JSON file or None """
    try:
        r = requests.get(url, timeout = 15, **kwargs)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f" [warn] {url[:80]} ---> {e}")
        return None

### Phillies

In [15]:
def get_phillies(seasons = SEASONS):
    rows = []
    for season in seasons:
        url = ( "https://statsapi.mlb.com/api/v1/schedule"
            f"?sportId=1&teamId=143&season={season}"
            "&gameType=R&fields=dates,date,games,"
            "teams,home,away,team,id,gameDate"
        )
        data = getter(url)
        if not data:
            continue
        for day in data.get("dates", []):
            for game in day.get("games", []):
                is_home = game["teams"]["home"]["team"]["id"] == 143
                if not is_home:
                    continue
                game_dt = pd.to_datetime(game["gameDate"], utc = True).tz_convert(eastern)
                rows.append({
                    "game_date": game_dt.date(),
                    "game_time": game_dt.strftime("%H:%M"),
                    "team": "Phillies",
                })
    return rows

### Flyers

In [16]:
# NHL seasons show up as "20252026" since they span calendar years
def get_flyers(seasons = SEASONS):
    rows = []
    for startyr in seasons:
        endyr = startyr + 1
        season_str = f"{startyr}{endyr}"
        url = f"https://api-web.nhle.com/v1/club-schedule-season/PHI/{season_str}"

        data = getter(url)
        if not data:
            continue
        for game in data.get("games", []):
            is_home = game.get("homeTeam", {}).get("abbrev") == "PHI"
            if not is_home:
                continue
            game_dt = pd.to_datetime(str(game["gameDate"]), utc = True).tz_convert(eastern)
            rows.append({
                "game_date": game_dt.date(),
                "game_time": game_dt.strftime("%H:%M"),
                "team": "Flyers",
            })

    return rows

### Eagles, Sixers
From same source (ESPN unofficial)

In [17]:
def get_espn(team_name, sport, league, abbrev, startyr, endyr):
    rows = []
    for season in range(startyr, endyr +1):
        url = (
            f"https://site.api.espn.com/apis/site/v2/sports"
            f"/{sport}/{league}/teams/{abbrev}/schedule?season={season}"
        )
        data = getter(url)
        if not data:
            continue
        for event in data.get("events", []):
            comp = event.get("competitions", [{}])[0]
            competitors = comp.get("competitors", [])
            if len(competitors) < 2:
                continue

            philly = next((c for c in competitors if abbrev.lower()
            in c.get("team", {}).get("abbreviation", "").lower()), None)
            if philly is None:
                continue

            is_home = philly.get("homeAway") == "home"
            if not is_home:
                continue
            game_dt = pd.to_datetime(event["date"], utc = True).tz_convert(eastern)

            rows.append({
                "game_date": game_dt.date(),
                "game_time": game_dt.strftime("%H:%M"),
                "team": team_name,
            })

    return rows

### Fetch, clean, and save

In [18]:
print("Fetching Phillies:")
rows_phillies = get_phillies()
print(f" {len(rows_phillies):,} games")

print("Fetching Flyers:")
rows_flyers = get_flyers()
print(f" {len(rows_flyers):,} games")

print("Fetching Eagles:")
rows_eagles = get_espn("Eagles", "football", "nfl", "phi", 2017, 2025)
print(f"  {len(rows_eagles):,} games")

print("Fetching Sixers:")
rows_sixers = get_espn("Sixers", "basketball", "nba", "phi", 2017, 2025)
print(f"  {len(rows_sixers):,} games")

Fetching Phillies:


 615 games
Fetching Flyers:


 344 games
Fetching Eagles:


  74 games
Fetching Sixers:


  360 games


In [19]:
df_sports = pd.DataFrame(rows_phillies + rows_flyers + rows_eagles + rows_sixers)
df_sports["game_date"] = pd.to_datetime(df_sports["game_date"])
df_sports["game_time"] = pd.to_datetime(df_sports["game_time"], format = "mixed").dt.strftime("%H:%M")
# sort on dates
df_sports = df_sports.sort_values("game_date").reset_index(drop = True)

# remove a few obs outside range of interest
df_sports = df_sports[(df_sports["game_date"] >= "2017-01-01") &
                      (df_sports["game_date"] <= "2026-01-01")]

print(f"\ndf_sports shape: {df_sports.shape}")
print(df_sports.groupby(["team"]).size().rename("n_games"))


df_sports shape: (1372, 3)
team
Eagles       73
Flyers      344
Phillies    615
Sixers      340
Name: n_games, dtype: int64


In [20]:
df_sports.to_parquet(f"{BASEPATH}/5_home_games.parquet", index = False)

In [21]:
df_sports = pd.read_parquet(f"{BASEPATH}/5_home_games.parquet")

### Prep for merge

In [22]:
# how many home games on this day
df_game_tally = (
    df_sports
    .groupby("game_date")
    .size()
    .rename("home_games")
    .reset_index()
)
df_game_tally

,game_date,home_games
0,2017-01-03,1
1,2017-01-11,1
2,2017-01-13,1
3,2017-01-18,1
4,2017-01-20,1
...,...,...
1202,2025-10-05,1
1203,2025-10-26,1
1204,2025-11-16,1
1205,2025-11-28,1


More complex merge, since we have game times and OTP records by the minute.
Build game windows as intervals (estimate game start time -> end time), within which we might expect to
see effects on RR performance.

In [23]:
# build game windows
# Estimated average length of games (Googled)
game_duration = {"Phillies": 3, "Flyers": 2.5, "Eagles": 3.5, "Sixers": 2.5}

df_sports["game_end"] = (
    pd.to_datetime(df_sports["game_time"],
                   format = "%H:%M") +
    df_sports["team"]
    .map(game_duration)
    .apply(lambda h: pd.Timedelta(hours = h))
    ).dt.strftime("%H:%M")

df_sports = df_sports.rename(columns = {"game_time": "game_start"})

In [24]:
# check if any games went past midnight
print(", ".join(sorted(df_sports["game_end"].unique())))

00:00, 00:25, 13:10, 14:35, 15:00, 15:05, 15:10, 15:30, 15:35, 15:40, 16:05, 16:10, 16:15, 16:30, 16:35, 17:00, 17:01, 17:05, 17:30, 17:35, 18:00, 18:05, 18:10, 18:30, 19:00, 19:05, 19:10, 19:15, 19:30, 19:35, 19:55, 20:00, 20:30, 21:00, 21:05, 21:20, 21:30, 21:35, 21:40, 21:45, 22:00, 22:05, 22:08, 22:10, 22:15, 22:30, 23:00, 23:45, 23:50


Some games did extend into 12a hour; `07_reduce_for_modeling.ipynb` handles game-window overlap directly from the saved `game_date`/`game_start`/`team` columns.

Build "windows" around game's start and (estimated) end times. These should rep periods of high
demand where we might expect to see stress on the system as people are commuting into/out
of the city before/after games. The windows are:
- `game_start` minus 90 minutes: captures commuters heading into the city ahead of game
- `game_start` plus 45 minutes: captures stragglers / attendees who may be late to arrive, and possible residual effects
- `game_end` minus 45 minutes: captures attendees who may be leaving a game early
- `game_end` plus 90 minutes: captures attendees taking trains out of city after game ends

In [25]:
def add_minutes(time_str, minutes):
    t = pd.to_datetime(time_str, format="%H:%M")
    t += pd.Timedelta(minutes = minutes)
    return t.strftime("%H:%M")

# Add 90m for travel to stadium before game start
# -- start time --
# Add 45m of extra padding after game start for stragglers
# -- game --
# Add 45m of extra padding before game end for early departures
# -- end time --
# Add 90m for travel from stadium after game end

df_sports["pre_start"]  = df_sports["game_start"].apply(lambda t: add_minutes(t, -90))
df_sports["post_start"]  = df_sports["game_start"].apply(lambda t: add_minutes(t, 45))

df_sports["pre_end"]  = df_sports["game_end"].apply(lambda t: add_minutes(t, -45))
df_sports["post_end"]   = df_sports["game_end"].apply(lambda t: add_minutes(t, 90))